In [ ]:
from matplotlib import pyplot as plt
import uproot
import mplhep as hep
from pathlib import Path

hep.style.use("CMS")

PLOT_DIR = Path("fits")
PLOT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
lumi_dict = {
    "2022": round(8077.0 / 1000, 2),
    "2022EE": round(27007.0 / 1000, 2),
    "2023": round(12231.77 / 1000, 2),
    "2023BPix": round(9451 / 1000, 2)
}

In [ ]:
for year in ("2022", "2022EE", "2023", "2023BPix"):
    file = uproot.open(f"efficiency_GloParT_{year}_QCD.root")

    hist = file["SF_TXbb"]
    poly = file["SF_TXbb_fit_eval"]

    fig, ax = plt.subplots(figsize=(12, 12))

    hist_values, hist_edges = hist.to_numpy()
    bin_centers = (hist_edges[:-1] + hist_edges[1:]) / 2
    bin_widths = hist_edges[1:] - hist_edges[:-1]
    
    # Calculate x errors (half the bin width)
    x_errors = bin_widths / 2

    # Plot histogram as points with error bars on both x and y
    hist_errors = hist.errors()
    ax.errorbar(bin_centers, hist_values, 
                xerr=x_errors, yerr=hist_errors, 
                fmt='o', color='blue', label='Histogram')

    # Extract polynomial/graph data - correct order
    poly_y, poly_x = poly.to_numpy()
    poly_errors = poly.errors()
    poly_x = (poly_x[:-1] + poly_x[1:]) / 2  # mid points

    # Plot polynomial/graph with error band
    ax.plot(poly_x, poly_y, 'r-', label='Fit')
    ax.fill_between(poly_x, poly_y - poly_errors, poly_y + poly_errors, 
                    color='blue', alpha=0.3, label='68% CL')

    # Add labels and legend
    ax.set_xlabel(r'$T_{Xbb}$')
    ax.set_ylabel('Scale Factor')
    ax.grid(True, alpha=0.3)
    ax.legend()

    hep.cms.label(ax=ax, label="Preliminary", lumi=lumi_dict[year], year=year, com=13.6)

    plt.tight_layout()
    plt.savefig(PLOT_DIR / f"SF_fit_{year}.pdf")
    plt.show()